# Lesson 6 - Creating an Healthcare Provider Agent using LangGraph, MCP, and A2A

In this lesson, you will build a third agent: a Healthcare Provider Agent. This agent demonstrates a powerful combination of technologies:

1.  **[MCP (Model Context Protocol)](https://modelcontextprotocol.io/)**: You will build a server that exposes a tool to find doctors from a JSON file.
2.  **[LangGraph]https://www.langchain.com/langgraph)**: You will build an agent that uses this MCP tool.
3.  **A2A**: You will wrap the LangGraph agent in an A2A server.
4.  **LiteLLM**: You will use LiteLLM to interact with the Google Gemini API.


## 6.1. Create the MCP Server

The MCP server code is provided in `mcpserver.py`. This server exposes a tool called `list_doctors` which queries a local `doctors.json` file.


In [1]:
from IPython.display import Code, display

display(Code("mcpserver.py"))

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

# Initialize the server
mcp = FastMCP("doctorserver")

# Load Data
# Adjusted path to match project structure
doctors: list = json.loads(Path("data/doctors.json").read_text())


@mcp.tool()
def list_doctors(state: str | None = None, city: str | None = None) -> list[dict]:
    """This tool returns a list of doctors practicing in a specific location. The search is case-insensitive.

    Args:
        state: The two-letter state code (e.g., "CA" for California).
        city: The name of the city or town (e.g., "Boston").

    Returns:
        A JSON string representing a list of doctors matching the criteria.
        If no criteria are provided, an error message is returned.
        Example: '[{"name": "Dr John James", "specialty": "Cardiology", ...}]'
    """
    # Input validation: ensure at least one search term is given.
    if not state and not city:
        return [{"error": "Please provide a state or a city."}]

    target_state = state.strip().lower() if state else None
    target_city = city.strip().lower() if city else None

    return [
        doc
        for doc in doctors
        if (not target_state or doc["address"]["state"].lower() == target_state)
        and (not target_city or doc["address"]["city"].lower() == target_city)
    ]


# Kick off server if file is run
if __name__ == "__main__":
    mcp.run(transport="stdio")

## 6.2. Test with LangGraph and MCP Client

Now you will verify that you can connect to the MCP server and use it within a LangChain/LangGraph agent. You use `MultiServerMCPClient` to connect to the local script via `stdio` transport.


In [2]:
import asyncio

from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_litellm import ChatLiteLLM
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.sessions import StdioConnection

from helpers import setup_env

setup_env()

In [3]:
mcp_client = MultiServerMCPClient(
    {
        "find_healthcare_providers": StdioConnection(
            transport="stdio",
            command="uv",
            args=["run", "mcpserver.py"],
        )
    }
)

In [4]:
agent = create_agent(
    ChatLiteLLM(
        model="gemini/gemini-3-flash-preview",
        # For Vertex AI:
        # model="vertex_ai/gemini-3-flash-preview",
        max_tokens=1000,
    ),
    asyncio.run(mcp_client.get_tools()),
    name="HealthcareProviderAgent",
    system_prompt="Your task is to find and list healthcare providers using the find_healthcare_providers MCP Tool based on the users query.",
)

In [5]:
prompt = "I'm based in Austin, TX. Are there any Psychiatrists near me?"
response = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    }
)

In [6]:
display(Markdown(response["messages"][-1].content))

Yes, there is a psychiatrist practicing in Austin, TX:

**Dr. Jessica Coffey**
*   **Specialty:** Psychiatry
*   **Address:** 1201 West 38th Street, Suite 210, Austin, TX 78705
*   **Phone:** (512) 555-0199
*   **Email:** j.coffey@austinmindhealth.com
*   **Experience:** 13 years
*   **Insurance Accepted:** Blue Cross Blue Shield, UnitedHealth, Aetna, Humana
*   **Accepting New Patients:** Yes

## 6.3. Wrap in A2A Server

We have encapsulated this logic in `a2a_provider_agent.py`. This file handles the MCP client connection and agent initialization and wraps the LangGraph agent as an A2A server using the [`langgraph-a2a-server`](https://github.com/5enxia/langgraph-a2a-server/) library.

In [8]:
from IPython.display import Code, display

display(Code("a2a_provider_agent.py"))

import asyncio
import os

from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)
from langchain.agents import create_agent
from langchain_litellm import ChatLiteLLM
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.sessions import StdioConnection
from langgraph.graph.state import CompiledStateGraph
from langgraph_a2a_server import A2AServer

from helpers import setup_env


def main() -> None:
    print("Running Healthcare Provider Agent")
    setup_env()

    HOST = os.getenv("AGENT_HOST", "localhost")
    PORT = int(os.getenv("PROVIDER_AGENT_PORT"))

    mcp_client = MultiServerMCPClient(
        {
            "find_healthcare_providers": StdioConnection(
                transport="stdio",
                command="uv",
                args=["run", "mcpserver.py"],
            )
        }
    )
    agent: CompiledStateGraph = create_agent(
        model=ChatLiteLLM(
            model="gemini/gemini-3-flash-preview",
            # For Vertex AI:
            # model="vertex_ai/gemini-3-flash-preview",
            max_tokens=1000,
        ),
        tools=asyncio.run(mcp_client.get_tools()),
        name="HealthcareProviderAgent",
        system_prompt="Find and list healthcare providers using the find_healthcare_providers MCP Tool.",
    )

    agent_card = AgentCard(
        name="HealthcareProviderAgent",
        description="Find healthcare providers by location and specialty.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[
            AgentSkill(
                id="find_healthcare_providers",
                name="Find Healthcare Providers",
                description="Finds providers based on location/specialty.",
                tags=["healthcare", "providers", "doctor", "psychiatrist"],
                examples=[
                    "Are there any Psychiatrists near me in Boston, MA?",
                    "Find a pediatrician in Springfield, IL.",
                ],
            )
        ],
    )

    server = A2AServer(
        graph=agent,
        agent_card=agent_card,
        host=HOST,
        port=PORT,
    )

    server.serve(app_type="starlette")


if __name__ == "__main__":
    main()

## 6.4. Run the Provider Server

Activate the Provider Agent in Terminal 3.
- Open a terminal as instructed below.
- Type `uv run a2a_provider_agent.py`

<div style="background-color:#e8f0fe; padding:15px; border-left:5px solid #4285f4; border-radius:4px">
    <b>Terminal Access:</b> Please open a new terminal window in your Jupyter environment to run the server.
    <br>You can typically do this by selecting <i>File -> New -> Terminal</i> from the menu.
</div>

## 6.7. Resources

- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://docs.langchain.com/oss/python/langchain/mcp)
- [LiteLLM](https://docs.litellm.ai/)


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>